# 第三章实验：协整与向量自回归

本实验共 **5 道题**，使用第三章的沪港指数数据和股价、利率、汇率数据。先运行各题中的基础示例，再按“作答要求”完成一个核心操作，填写简短代码和结论。

| 题目 | 主要内容 | 专业接口 |
|---|---|---|
| 1 | 数据变换与平稳性 | `adfuller` |
| 2 | VAR 估计、诊断与预测 | `VAR`、`select_order`、`test_whiteness`、`forecast` |
| 3 | 协整检验与误差修正 | `coint`、`select_coint_rank`、`VECM` |
| 4 | Granger 预测关系 | `test_causality` |
| 5 | 冲击响应与方差分解 | `irf`、`fevd` |

**运行方式：**用 PyCharm、VS Code 或 JupyterLab 打开本文件，选择 Python 内核，从头运行。`Shift+Enter` 通常用于运行当前单元格。重启内核后，先运行前面的环境与数据单元格，再运行后续代码。

代码输出、表格字段和图表标签使用英文，正文和注释使用中文。价格、收益率、差分值是不同的变量，解读结果时须同时说明变量、单位和样本。

## 环境与文件

学生版 Notebook 与 `VARSJ.csv`、`XZSJ.csv` 放在同一文件夹。教师版使用同级“学生材料”中的数据。代码只用相对路径，整体复制文件夹后无需填写新的盘符。

首个代码单元格只补装缺少的库，不主动升级已有库。补装需要联网，使用的是**当前 Notebook 内核**的 Python。安装后若仍出现导入问题，重启内核并从头运行。Notebook 尚不能运行任何单元格时，应先在编辑器中选好 Python 内核。

In [ ]:
import sys
import subprocess
import importlib
import warnings

# 实际尝试导入；缺少的包用当前内核安装，避免装进其他 Python 环境。
packages = ("numpy", "pandas", "scipy", "matplotlib", "statsmodels")
missing = []
for name in packages:
    try:
        importlib.import_module(name)
    except ModuleNotFoundError:
        missing.append(name)
if missing:
    print("Installing missing packages:", ", ".join(missing), flush=True)
    try:
        importlib.import_module("pip")
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "ensurepip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()

from pathlib import Path
from inspect import signature
from functools import partial
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.vecm import select_coint_rank, coint_johansen, VECM
from numpy.exceptions import ComplexWarning
from IPython.display import display
%matplotlib inline

# statsmodels 0.15 新增返回对象选项；这里保持 ADF 的元组接口。
if "result_object" in signature(adfuller).parameters:
    adfuller = partial(adfuller, result_object=False)

# 支持从学生材料、教师用或实验课目录启动；不搜索其他位置。
folders = (Path("."), Path("学生材料"), Path("../学生材料"))
DATA_DIR = next(
    (folder for folder in folders
     if all((folder / name).is_file() for name in ("VARSJ.csv", "XZSJ.csv"))),
    None
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Keep VARSJ.csv and XZSJ.csv in the student materials folder. "
        "Start the notebook kernel in the notebook folder or the lab folder."
    )
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.unicode_minus": False,
})
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "pandas:", pd.__version__, "statsmodels:", statsmodels.__version__)
print("Imports successful. Both data files were found.")

### 独立作答区

每题的代码作答区以 `%%practice q1` 等标记开头。保留标记，在下面编写代码；准备好后改为 `%%practice q1 --run` 并执行。删除 `--run` 可恢复跳过。

- 每题取得数据副本，可使用 `prices`、`log_prices`、`returns`、`train`、`test`、`levels`、`H` 及已导入的专业函数。每次执行都重新开始，模型需要在本题中重新拟合。
- 用 `display(...)` 显示表格、`plt.show()` 显示图形；末行表达式也会显示。
- 通常的语法和计算错误只显示在本题下方，不中断后续基础示例；普通的数据赋值不改变其他题的数据。运行耗时循环时可用编辑器的停止按钮中断。
- `skipped` 表示未运行，`failed` 表示出错，`completed` 表示执行完毕。执行成功仍须检查结果与统计解释。

下面折叠的代码用于注册作答区，按顺序运行即可。

In [ ]:
import ast
import traceback
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

practice_status = {}

@register_cell_magic
def practice(line, cell):
    parts = line.split()
    if not parts or parts[0] not in {"q1", "q2", "q3", "q4", "q5"}:
        print("Use an exercise label from q1 to q5.")
        return
    label = parts[0]
    if len(parts) not in (1, 2) or (len(parts) == 2 and parts[1] != "--run"):
        print("Usage: %%practice q1 --run")
        return
    if "--run" not in parts:
        practice_status[label] = "skipped"
        print(f"Exercise {label} skipped. Add --run when ready.")
        return
    main = get_ipython().user_ns
    names = ("np", "pd", "plt", "display", "VAR", "adfuller", "coint",
             "select_coint_rank", "johansen_rank", "VECM", "adf_table")
    scope = {name: main[name] for name in names if name in main}
    for name in ("prices", "log_prices", "returns", "train", "test", "levels", "H"):
        if name in main:
            value = main[name]
            scope[name] = value.copy(deep=True) if isinstance(value, (pd.Series, pd.DataFrame)) else value
    figures_before = set(plt.get_fignums())
    try:
        with plt.rc_context():
            filename = f"<exercise {label}>"
            tree = ast.parse(cell, filename=filename)
            if not tree.body:
                practice_status[label] = "empty"
                print(f"Exercise {label} has no code yet.")
                return
            last = tree.body.pop() if isinstance(tree.body[-1], ast.Expr) else None
            exec(compile(tree, filename, "exec"), scope)
            if last is not None:
                value = eval(compile(ast.Expression(last.value), filename, "eval"), scope)
                if value is not None:
                    display(value)
            plt.show()
        practice_status[label] = "completed"
        print(f"Exercise {label} completed.")
    except (Exception, SystemExit):
        practice_status[label] = "failed"
        traceback.print_exc(limit=2)
        print(f"Exercise {label} failed. Revise this cell and try again.")
    finally:
        for number in set(plt.get_fignums()) - figures_before:
            plt.close(number)

print("Practice cells are ready.")

## 第 1 题：数据变换与平稳性

`VARSJ.csv` 包含上证综指（SZH）与香港恒生指数（XGH）的价格。文件第一列为日期，使用 GBK 编码。两市场已在同一行对齐，按现有观测顺序分析，不补入非交易日。

`XZSJ.csv` 用于第 3 题。SZH 是**已经取自然对数**的上证综指，SHIBOR 按课件口径为 7 天 Shibor，EX 为人民币元／美元汇率。不能再次对该文件的 SZH 取对数。

价格数据转换为对数价格与百分数收益率：

$$
z_t=\ln P_t,\qquad r_t=100(z_t-z_{t-1}).
$$

`r=1.0` 表示对数收益率 1%，收益率预测误差的单位为百分点。给定文件没有完整的复权与分红元数据，本实验按文件中的指数数值分析。

最后 20 个共同交易观测作为 VAR 预测的留出期，之前的数据用于检验、选阶和估计。`XZSJ.csv` 则独立用于样本内协整分析，不用于选择沪港收益率预测模型。

In [ ]:
# 先统一日期与数值类型，再检查重复记录、缺失值和非正价格。
raw = pd.read_csv(DATA_DIR / "VARSJ.csv", encoding="gbk")
prices = raw[["SZH", "XGH"]].apply(pd.to_numeric)
prices.index = pd.to_datetime(raw.iloc[:, 0])
prices.index.name = "Date"
prices = prices.sort_index()

levels = pd.read_csv(DATA_DIR / "XZSJ.csv", parse_dates=["Time"])
levels = levels.set_index("Time")[["SZH", "SHIBOR", "EX"]].apply(pd.to_numeric).sort_index()
levels.index.name = "Date"
for name, frame in (("Prices", prices), ("Levels", levels)):
    if frame.index.hasnans or frame.index.has_duplicates:
        raise ValueError(f"{name}: dates must be valid and unique.")
    if not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f"{name}: all observations must be finite.")
if (prices <= 0).any().any():
    raise ValueError("Prices must be positive before taking logarithms.")

log_prices = np.log(prices)
returns = 100 * log_prices.diff().dropna()
H = 20
train, test = returns.iloc[:-H].copy(), returns.iloc[-H:].copy()
if len(train) < 500:
    raise ValueError("At least 500 training returns are required.")

sample_rows = []
for name, frame in (("Prices", prices), ("VAR training", train),
                    ("VAR holdout", test), ("Cointegration levels", levels)):
    sample_rows.append({"Dataset": name, "Rows": len(frame),
                        "Start": str(frame.index[0].date()), "End": str(frame.index[-1].date())})
display(pd.DataFrame(sample_rows).set_index("Dataset"))
display(prices.head(3))

In [ ]:
# 对数价格与收益率分开画，不把不同尺度混在同一纵轴。
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
log_prices.loc[:train.index[-1]].plot(ax=axes[0], lw=0.9)
axes[0].set(title="Training sample: log prices", ylabel="Log index")
train.plot(ax=axes[1], lw=0.65)
axes[1].axhline(0, color="gray", lw=0.7)
axes[1].set(title="Training sample: log returns", ylabel="Daily log return (%)", xlabel="Date")
fig.tight_layout()
plt.show()

### ADF 的输入与输出

`adfuller` 每次检验一列序列，原假设是“存在单位根”。`regression="c"` 含常数，`"ct"` 含常数与线性趋势；`maxlag=12` 给出差分滞后的候选上限，`autolag="AIC"` 在其中选阶。

下面用一个短函数整理重复的检验结果。`ADF` 是统计量，`p_value` 是检验的 $p$ 值，`Lags` 是实际选取的差分滞后数。小于 0.05 时在 5% 水平拒绝单位根；未拒绝不能表述为已经证明存在单位根。

In [ ]:
def adf_table(frame, regression="c", maxlag=12):
    rows = []
    for name in frame.columns:
        result = adfuller(frame[name].dropna(), regression=regression,
                          maxlag=maxlag, autolag="AIC")
        rows.append({"Series": name, "Deterministic": regression,
                     "ADF": result[0], "p_value": result[1],
                     "Lags": result[2], "N_obs": result[3]})
    return pd.DataFrame(rows).set_index("Series")

z_train = log_prices.loc[:train.index[-1]]
adf_prices = adf_table(z_train)
adf_returns = adf_table(train)
display(pd.concat({"Log price": adf_prices, "Log return": adf_returns},
                  names=["Transform", "Series"]))

**结果阅读：**本样本在上述设定下，对数价格未拒绝单位根，收益率拒绝单位根，因此后续稳定 VAR 使用收益率。ADF 不检验残差是否白噪声，也不能证明收益率独立、正态或方差恒定。

### 作答要求

使用完整训练期的 SZH 对数价格与收益率，采用含常数的 ADF 检验（`regression="c"`、`maxlag=12`、`autolag="AIC"`）。报告两项 p 值，并据此说明哪个序列更适合直接进入稳定 VAR。

In [ ]:
%%practice q1
# 可用 log_prices、train 和 adf_table(...)。
# 对数价格截取到 train 的最后一个日期，只选 SZH 列。
# adf_table 默认使用 maxlag=12、autolag="AIC"。

**第 1 题分析：**

填写两个 p 值，以及据此选择哪一种序列。

## 第 2 题：VAR 估计、诊断与预测

二元 VAR 同时用两市场的过去值解释当期收益率。`VAR` 的输入为“行是时间、列是变量”的二维数据。调用 `fit(p)` 后，`p` 是每个变量共同使用的滞后阶数。

`select_order(12)` 比较 0—12 阶。信息准则越小越优，但各准则可能选出不同阶数。0 阶是只含常数的模型。本例 AIC 与 FPE 选择 7 阶，BIC 与 HQIC 选择 0 阶，需结合残差检查讨论。

模型输入重置为连续整数索引，可避免软件推断不规则交易日频率；原日期仍保存在 `train` 和 `test` 中。

In [ ]:
model = VAR(train.reset_index(drop=True))
selection = model.select_order(maxlags=12)
ic_table = pd.DataFrame(selection.ics).rename_axis("Lag")
display(ic_table)
print("Selected lags:", {name: int(value) for name, value in selection.selected_orders.items()})

# 分别观察简单模型与 AIC 候选；每个模型只拟合一次，后续复用。
fits = {p: model.fit(p, trend="c") for p in (1, 7)}
diagnostics = []
for p, fit in fits.items():
    white = fit.test_whiteness(nlags=12)
    diagnostics.append({"Lag": p, "AIC_common": ic_table.loc[p, "aic"],
                        "BIC_common": ic_table.loc[p, "bic"],
                        "Stable": fit.is_stable(), "White_p": white.pvalue})
display(pd.DataFrame(diagnostics).set_index("Lag"))

fit_var = fits[7]  # 此数据的 7 阶候选通过稳定性和 12 阶残差诊断。
display(fit_var.params.head(5))

### 诊断和系数的读法

- `is_stable()` 判断动态稳定性。`fit_var.roots` 返回滞后多项式的根，稳定模型的有限根在单位圆**外**。
- `test_whiteness(nlags=12)` 检验前 12 阶的残差自相关和交叉相关是否为零。`nlags` 必须大于 VAR 阶数；`White_p` 小于 0.05 说明仍有遗漏的动态联系。
- 参数表中**列是被解释变量，行是解释变量**。`L1.SZH` 行、`XGH` 列是上海收益率一期滞后对香港收益率条件均值的系数，不是已识别的经济因果效应。

在当前训练样本中，VAR(1) 稳定，但未通过残差白噪声检验；VAR(7) 在该检验下未被拒绝。后面的 Granger、IRF 和 FEVD 基础示例使用 VAR(7)。

### 固定起点预测

`forecast(y, steps=H)` 需要最后 `p` 行历史观测。下面只从训练期末一次预测 20 步，预测期间不读入实际留出收益率。预测生成后才与 `test` 对齐，并与零收益率基准比较。

In [ ]:
# last_p 的列顺序必须与模型拟合时相同。
last_p = train.to_numpy()[-fit_var.k_ar:]
forecast = pd.DataFrame(fit_var.forecast(last_p, steps=H),
                        index=test.index, columns=test.columns)
metrics = []
for label, prediction in (("VAR(7)", forecast),
                          ("Zero return", pd.DataFrame(0.0, index=test.index, columns=test.columns))):
    errors = prediction - test
    for name in test.columns:
        metrics.append({"Model": label, "Series": name,
                        "MAE": errors[name].abs().mean(),
                        "RMSE": np.sqrt(errors[name].pow(2).mean())})
forecast_metrics = pd.DataFrame(metrics).set_index(["Model", "Series"])
display(forecast_metrics)

fig, axes = plt.subplots(2, 1, figsize=(10, 4.8), sharex=True)
for ax, name in zip(axes, test.columns):
    ax.plot(test.index, test[name], "o-", ms=3, label="Actual")
    ax.plot(forecast.index, forecast[name], color="#E07A2D", label="VAR(7)")
    ax.axhline(0, color="gray", ls="--", label="Zero return")
    ax.set(title=name, ylabel="Log return (%)")
    ax.legend(loc="best", ncol=3)
fig.tight_layout()
plt.show()

**结果阅读：**MAE、RMSE 的单位都是百分点。在本次 20 期留出样本中，VAR(7) 对上海收益率的 RMSE 略低于零收益率基准，对香港收益率则略高。检验合格与样本内拟合改善，并不保证样本外优于简单基准。不能再根据这 20 期结果反复调阶并称其为独立检验。

### 作答要求

使用 `train` 拟合 VAR(7)，预测留出期第一个交易观测的 SZH 和 XGH 收益率。给出这一期的两个点预测，单位为百分数。

In [ ]:
%%practice q2
# 可用 train、test 和 VAR。
# forecast 的历史输入是训练期最后 p 行，steps=1 表示预测一步。
# 输出日期可使用 test.index[:1]。

**第 2 题分析：**

填写预测日期及两市场的一步收益率预测值（%）。

## 第 3 题：协整与 VECM

本题使用 `levels`，即 `XZSJ.csv` 中的 SZH、SHIBOR、EX。水平序列和差分序列的 ADF 检验用于判断是否适合按 $I(1)$ 系统分析；协整检验随后判断是否存在平稳的长期线性组合。

**Engle–Granger：**`coint(y, x)` 的原假设为“没有协整关系”。该函数使用残差协整检验的专用分布，不能用对估计残差直接调用普通 ADF 得到的 p 值替代。

In [ ]:
# SZH 已取对数；本题保持三个变量在文件中的尺度。
display(pd.concat({
    "Level": adf_table(levels, maxlag=11),
    "First difference": adf_table(levels.diff().dropna(), maxlag=11),
}, names=["Transform", "Series"]))

eg_stat, eg_p, eg_critical = coint(
    levels["SZH"], levels["SHIBOR"], trend="c",
    maxlag=11, autolag="AIC", return_results=False
)
display(pd.DataFrame([{"Pair": "SZH, SHIBOR", "Statistic": eg_stat,
                       "p_value": eg_p, "Critical_5pct": eg_critical[1]}]))

### Johansen 协整秩

`select_coint_rank` 的输入仍是**水平数据**，不能先把它们差分。`k_ar_diff=q` 指 VECM 中差分项的滞后数，对应水平 VAR 的阶数 $p=q+1$。

`method="trace"` 使用迹检验，`signif=0.05` 对应 5% 临界值，`det_order=0` 为该检验接口中的常数项设定。检验从“秩为 0”开始，依次检验“秩不超过 1”等假设，在首次不拒绝处停止。

| 所选秩 $r$ | 对三个 $I(1)$ 变量的解释 |
|---|---|
| $r=0$ | 未发现协整，考虑差分 VAR |
| $0<r<3$ | 可估计含 $r$ 条协整关系的 VECM，再检查残差 |
| $r=3$ | 应重新检查单整判断和设定；满足稳定性条件时考虑水平 VAR |

下面的 `johansen_rank` 统一使用常数项、迹检验和 5% 水平，内部调用 `select_coint_rank`。辅助检查用于兼容部分 NumPy 版本返回的复数类型：仅在确认特征值可视为实数后继续，实际存在非零虚部或非有限值时停止。

In [ ]:
def johansen_rank(frame, k_ar_diff):
    # 检验理论中的特征值应为实数；只处理这处已检查的类型转换提示。
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message="Casting complex values to real discards the imaginary part",
            category=ComplexWarning, module=r"statsmodels\.tsa\.vector_ar\.vecm"
        )
        eigenvalues = coint_johansen(frame, det_order=0, k_ar_diff=k_ar_diff).eig
        if not np.isfinite(eigenvalues).all() or np.iscomplexobj(np.real_if_close(eigenvalues)):
            raise ValueError("Johansen eigenvalues must be finite and real. Check the data and lags.")
        return select_coint_rank(frame, det_order=0, k_ar_diff=k_ar_diff,
                                 method="trace", signif=0.05)

q = 1  # 差分滞后 1 阶，对应水平 VAR(2)，不是 VAR(1)。
rank_result = johansen_rank(levels, k_ar_diff=q)
print(rank_result.summary())
print("Selected rank:", rank_result.rank)

### VECM 的估计与诊断

VECM 将短期变化写为

$$
\Delta \mathbf y_t=\mathbf c+\alpha\beta'\mathbf y_{t-1}
+\sum_{i=1}^{q}\Gamma_i\Delta\mathbf y_{t-i}+\mathbf u_t.
$$

`beta` 的每一列是一条协整向量，`alpha` 的对应列给出各变量的调整系数。解释 `alpha` 时，须同时看 `beta` 的符号和归一化；有多条关系时，还须保持其他误差修正项不变。

`deterministic="co"` 表示常数放在协整关系外，`"ci"` 表示限制在关系内。`det_order` 和 `deterministic` 是两个不同接口的设定，不能按编码机械互换。本例沿用课件“常数项检验＋关系外常数估计”的演示组合。

In [ ]:
rank = int(rank_result.rank)
if 0 < rank < levels.shape[1]:
    # fit() 输入水平数据，模型内部构造差分和误差修正项。
    vecm_fit = VECM(levels.reset_index(drop=True), k_ar_diff=q,
                    coint_rank=rank, deterministic="co").fit()
    relation_names = [f"EC{i + 1}" for i in range(rank)]
    display(pd.DataFrame(vecm_fit.beta, index=levels.columns, columns=relation_names)
            .rename_axis("Beta"))
    display(pd.DataFrame(vecm_fit.alpha, index=levels.columns, columns=relation_names)
            .rename_axis("Alpha"))
    vecm_white = vecm_fit.test_whiteness(nlags=12)
    print("VECM residual whiteness p-value:", vecm_white.pvalue)
    print("Residual check:", "Not rejected" if vecm_white.pvalue >= 0.05 else "Rejected")
else:
    print("No reduced-rank VECM is fitted for this rank.")

**结果阅读：**当前二元 Engle–Granger 检验没有拒绝“不存在协整”，三变量 Johansen 检验在 `q=1` 时选择秩 2，两者的变量集合不同。该 VECM 的残差白噪声检验 p 值远小于 0.05，因此不能仅凭“秩为 2”就把长期关系作为稳健结论。本题继续改变滞后数，观察结论是否保持。

接口：[coint](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.coint.html)、[select_coint_rank](https://www.statsmodels.org/stable/generated/statsmodels.tsa.vector_ar.vecm.select_coint_rank.html)、[VECM](https://www.statsmodels.org/stable/generated/statsmodels.tsa.vector_ar.vecm.VECM.html)。

### 作答要求

对 `levels` 调用 `johansen_rank`，设 `k_ar_diff=2`。读出所选协整秩，并指出迹检验在哪一行首次不拒绝原假设。

In [ ]:
%%practice q3
# 可用 levels 和 johansen_rank。
# summary() 查看顺序检验表，rank 读取选定的协整秩。

**第 3 题分析：**

填写所选协整秩，以及首次不拒绝的原假设。

## 第 4 题：Granger 预测关系

检验 SZH 是否为 XGH 的 Granger 原因时，考察 XGH 方程中 **SZH 的全部滞后系数是否共同为零**：

$$
H_0:b_1=\cdots=b_p=0.
$$

受限模型把这些系数设为零，无约束模型允许其自由估计。`test_causality(caused, causing)` 的第一个参数是被预测变量，第二个参数是新增信息来源。因此 `test_causality("XGH", ["SZH"])` 检验方向是 SZH → XGH。

小 p 值反对“没有新增预测信息”，并不证明经济干预的因果效应。反方向须另做检验；某一方向显著不意味着另一方向也显著。

In [ ]:
granger_rows = []
for cause, response in (("SZH", "XGH"), ("XGH", "SZH")):
    result = fit_var.test_causality(caused=response, causing=[cause], kind="f")
    granger_rows.append({"Direction": f"{cause} -> {response}",
                         "VAR_lag": fit_var.k_ar, "F": result.test_statistic,
                         "p_value": result.pvalue, "Reject_5pct": result.pvalue < 0.05})
display(pd.DataFrame(granger_rows).set_index("Direction"))

**结果阅读：**VAR(7) 在这段训练样本中两个方向的 p 值均高于 0.05，应表述为“在该变量集合、滞后阶数和样本下，未发现 5% 水平的 Granger 预测作用”。不能改写为两个市场互不相关，或绝不存在任何联系。

接口：[VARResults.test_causality](https://www.statsmodels.org/stable/generated/statsmodels.tsa.vector_ar.var_model.VARResults.test_causality.html)。该函数的系统 F 检验自由度口径可能与单方程手工公式不同。

### 作答要求

使用 `train` 拟合 VAR(7)，检验 SZH 是否对 XGH 提供 Granger 预测信息。报告该方向的 F 检验 p 值，并给出 5% 水平下的结论。

In [ ]:
%%practice q4
# 可用 train 和 VAR。
# test_causality 中 caused 是被预测变量，causing 是待检验的信息来源。

**第 4 题分析：**

填写 p 值，并给出这个方向的检验结论。

## 第 5 题：脉冲响应与方差分解

本题继续使用收益率 VAR(7)。IRF 描述一次冲击后各期的响应，FEVD 衡量预测误差方差中各冲击的贡献。二者均依赖冲击识别方式。

默认 Cholesky 顺序为 `["SZH", "XGH"]`。第一个正交化冲击当期可影响两变量，第二个冲击对第一个变量的当期影响为零。冲击按一个标准差规范化，并不表示观测到的收益率恰好上升一个百分点。

| 输出 | 维度顺序 | 第一个期限 |
|---|---|---|
| `irf.orth_irfs` | 响应期限、响应变量、冲击来源 | 第 0 项为冲击当期 $h=0$ |
| `fit.fevd(10).decomp` | 响应变量、预测期限、冲击来源 | 第 0 项为一步预测 $H=1$ |

在当前列顺序下，`orth_irfs[:, 1, 0]` 是 SZH 冲击对 XGH 的响应。图中的区间采用渐近标准误形成逐期 95% 置信区间，属于模型假设下的近似结果，不能视为整条路径的同时置信带。

In [ ]:
irf = fit_var.irf(periods=10)
orth = irf.orth_irfs
se = irf.stderr(orth=True)
horizons = np.arange(11)  # IRF 包含冲击当期 h=0，共 11 个时点。

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, (response, shock) in zip(axes, ((1, 0), (0, 1))):
    values = orth[:, response, shock]
    error = 1.96 * se[:, response, shock]
    ax.plot(horizons, values, color="#003366", label="Response")
    ax.fill_between(horizons, values - error, values + error,
                    color="#003366", alpha=0.15, label="95% pointwise CI")
    ax.axhline(0, color="gray", lw=0.8)
    ax.set(title=f"{fit_var.names[shock]} shock -> {fit_var.names[response]}",
           xlabel="Horizon (trading observations)", ylabel="Response (percentage points)")
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
decomp = fit_var.fevd(periods=10).decomp
fevd_rows = []
for response, name in enumerate(fit_var.names):
    for horizon in (1, 5, 10):
        shares = decomp[response, horizon - 1, :] * 100
        fevd_rows.append({"Response": name, "Horizon": horizon,
                          "SZH_shock_pct": shares[0], "XGH_shock_pct": shares[1]})
display(pd.DataFrame(fevd_rows).set_index(["Response", "Horizon"]))
print("Shares sum to 100 percent:", np.allclose(decomp.sum(axis=2), 1))

**结果阅读：**原顺序下，SZH 冲击约占 XGH 一步预测误差方差的 32.29%，这不是香港收益率“有 32.29% 由上海决定”，也不是回归的 $R^2$。FEVD 使用响应的平方，不表示正向还是负向影响；方向与不确定性应查看 IRF。

交换列顺序会改变 Cholesky 当期约束。比较时必须按**变量名称**重新查找响应和冲击的位置，不能沿用原来的数字索引。重新排序改变的是冲击定义，相应占比差异不能直接解释为经济联系发生了变化。

接口：[VAR 的估计、预测与冲击分析](https://www.statsmodels.org/stable/vector_ar.html)、[IRF 标准误](https://www.statsmodels.org/stable/generated/statsmodels.tsa.vector_ar.irf.IRAnalysis.stderr.html)。

### 作答要求

固定变量顺序为 `["SZH", "XGH"]`，使用 `train` 拟合 VAR(7)。读取 SZH 冲击对 XGH 在 **10 步预测误差方差**中的贡献，报告百分比，并用一句话解释其含义。

In [ ]:
%%practice q5
# 可用 train 和 VAR，保持题目指定的列顺序。
# fevd(...).decomp 的维度依次为：响应变量、预测期限、冲击来源。
# 期限数组从索引 0 开始，读取后乘以 100 得到百分比。

**第 5 题分析：**

填写 10 步贡献比例，并解释这个比例衡量什么。

## 完成与保存

本实验共 5 题。完成各题的代码和文字分析后，保留对应的 `--run`，重启内核并从头运行。确认每题显示 `completed`，检查结果是否满足题意，再保存含输出的 Notebook。

VAR 模型的选阶和估计只用训练样本。协整分析须联合考察单整阶数、确定性项、滞后和残差。统计检验结果、预测误差和冲击解释分别回答不同问题，结论应与采用的模型和数据范围相对应。

In [ ]:
# 状态表只检查作答代码是否运行，不代替统计内容的评分。
display(pd.Series({f"q{i}": practice_status.get(f"q{i}", "not run")
                   for i in range(1, 6)}, name="Status").rename_axis("Exercise").to_frame())